# 08. Submission Pipeline

This notebook builds and validates competition submissions from saved experiment predictions.

The first submission candidate uses:

- 50% CatBoost probabilities
- 50% XGBoost probabilities
- decision threshold: 0.130

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
EXPERIMENTS_DIR = PROJECT_ROOT / "artifacts" / "experiments"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"

COMPARISON_DIR = EXPERIMENTS_DIR / "comparison_v0_raw_minimal"

BLEND_PREDICTIONS_PATH = COMPARISON_DIR / "selected_blend_test_predictions.parquet"

SAMPLE_SUBMISSION_PATH = DATA_DIR / "raw" / "final_proj_sample_submission.csv"

SUBMISSIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Project root:          {PROJECT_ROOT}")
print(f"Comparison directory: {COMPARISON_DIR}")
print(f"Submissions directory:{SUBMISSIONS_DIR}")

Project root:          C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final
Comparison directory: C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\comparison_v0_raw_minimal
Submissions directory:C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\submissions


In [2]:
if not BLEND_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(f"Blend predictions not found: {BLEND_PREDICTIONS_PATH}")

if not SAMPLE_SUBMISSION_PATH.exists():
    raise FileNotFoundError(f"Sample submission not found: {SAMPLE_SUBMISSION_PATH}")

blend_predictions = pd.read_parquet(BLEND_PREDICTIONS_PATH)

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(
    "Blend predictions:",
    blend_predictions.shape,
)

print(
    "Sample submission:",
    sample_submission.shape,
)

print(
    "Sample submission columns:",
    sample_submission.columns.tolist(),
)

display(blend_predictions.head())
display(sample_submission.head())

Blend predictions: (2500, 3)
Sample submission: (2500, 2)
Sample submission columns: ['index', 'y']


,row_index,probability,prediction
0,0,0.001349,0
1,1,0.030158,0
2,2,0.005894,0
3,3,0.009900,0
4,4,0.004379,0


,index,y
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [3]:
required_prediction_columns = {
    "row_index",
    "probability",
    "prediction",
}

missing_prediction_columns = required_prediction_columns - set(
    blend_predictions.columns
)

if missing_prediction_columns:
    raise ValueError(
        f"Missing prediction columns: {sorted(missing_prediction_columns)}"
    )

if len(blend_predictions) != len(sample_submission):
    raise ValueError(
        "Prediction row count does not match "
        "sample submission row count: "
        f"{len(blend_predictions)} != "
        f"{len(sample_submission)}"
    )

if blend_predictions["row_index"].duplicated().any():
    raise ValueError("Blend predictions contain duplicate row indices.")

expected_row_indices = pd.Series(
    range(len(blend_predictions)),
    name="row_index",
)

actual_row_indices = blend_predictions["row_index"].reset_index(drop=True)

if not actual_row_indices.equals(expected_row_indices):
    raise ValueError("Blend prediction rows are not in the expected 0..n-1 order.")

if not blend_predictions["probability"].between(0.0, 1.0).all():
    raise ValueError("Probabilities must be between 0 and 1.")

if not set(blend_predictions["prediction"].unique()).issubset({0, 1}):
    raise ValueError("Predictions must contain only classes 0 and 1.")

print("Initial submission validation passed.")

Initial submission validation passed.
